In [3]:
import torch

from dataset.LPIdataset import LPIDataset
from torch.utils.data import DataLoader
from models.CNN import UNet
from models._config import *

def load_models(ckpt, model):
    state_dict = torch.load(ckpt)
    new_state_dict = {}
    for k, v in state_dict.items():
        nk = k.replace('module.', '')
        if 'query' not in nk:  # Remove "query" from state_dict
            new_state_dict[nk] = v
    model.load_state_dict(new_state_dict, strict=False)  # Allow partial loading
    model.eval()
    return model


class UNet_lrp(UNet):
    
    def lrp(self, x, target_class):
        e1, e2, e3, e4, m, d4, d3, d2, d1 = self.forward(x, return_activations=True)

        logits = self.forward(x)  # Full forward pass for logits
        R = torch.zeros_like(logits)
        R[:, target_class] = logits[:, target_class]

        # Backpropagate through fully connected layer
        R = self._lrp_fc(self.output, self.global_pool(m), R)

        # Backpropagate through decoders
        R = self._lrp_decoder(self.decoder1, e1, d1, R)
        R = self._lrp_decoder(self.decoder2, e2, d2, R)
        R = self._lrp_decoder(self.decoder3, e3, d3, R)
        R = self._lrp_decoder(self.decoder4, e4, d4, R)

        # Backpropagate through middle and encoders
        R = self._lrp_conv(self.middle, m, e4, R)
        R = self._lrp_conv(self.encoder4, e4, e3, R)
        R = self._lrp_conv(self.encoder3, e3, e2, R)
        R = self._lrp_conv(self.encoder2, e2, e1, R)
        R = self._lrp_conv(self.encoder1, e1, x, R)

        return R

ckpt = '/home/kiwan/TSC_XAI/ckpts/pwnNoisy/_best_0.0134.pth'
model = UNet_lrp(in_channels=1, out_channels=len(waveforms))
model = load_models(ckpt, model)

# dataset = '/data/kiwan/LPI_KIWAN_STFT/'
dataset = '/home/kiwan/TSC_XAI/dataset/lpi_STFTset'
dataload = LPIDataset(dataset, waveforms, model_type='UNet')



In [4]:
import torch
from torch.utils.data import DataLoader
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

batch_size = 32
dataloader = DataLoader(dataload, batch_size=batch_size, shuffle=False, collate_fn=collate)


model = model.to(device)



def calculate_accuracy(model, dataloader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels, _, _, _ in dataloader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).int().sum().item()
            total += batch_size

    accuracy = correct / total * 100
    return accuracy

import matplotlib.pyplot as plt
import torchvision.transforms as transforms

# Transform to convert tensor to numpy
to_numpy = transforms.ToPILImage()

def plot_lrp_results(model, dataloader, target_class):
    """
    Plot original images and corresponding LRP results with a jet colormap.
    """
    model.eval()

    with torch.no_grad():
        for images, labels, _, _, _ in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            # Compute LRP for the batch
            for i in range(len(images)):
                image = images[i:i+1]  # Select one image at a time
                original_image = image.squeeze().cpu().numpy()  # Extract the original image
                relevance = model.lrp(image, target_class).squeeze().cpu().numpy()  # LRP result

                # Plotting
                fig, axes = plt.subplots(1, 2, figsize=(10, 5))
                axes[0].imshow(original_image, cmap='gray')
                axes[0].set_title("Original Image")
                axes[0].axis('off')

                axes[1].imshow(relevance, cmap='jet')
                axes[1].set_title("LRP Heatmap")
                axes[1].axis('off')

                plt.show()

                # To plot a single image, break here; remove for all images
                break
            break  # Remove this to process all batches


# Plot LRP results for target class
target_class = 0  # Change to the desired target class
plot_lrp_results(model, dataloader, target_class)

# accuracy = calculate_accuracy(model, dataloader)
# print(f"Model Accuracy: {accuracy:.2f}%")
# 

TypeError: UNet.forward() got an unexpected keyword argument 'return_activations'